<a href="https://colab.research.google.com/github/abhishek18-blog/Abstracts-researchHub/blob/main/konkanCoast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# Cell 1
from google.colab import auth
import ee
import tensorflow as tf

# 1. Use Colab's native authentication first (this usually bypasses pop-up blockers)
auth.authenticate_user()

# 2. Then initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='earth-482204')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [26]:
import ee

# 1. Initialize (Authentication is already done!)
ee.Initialize(project='earth-482204') # Using your whitelisted project ID

# 2. Define our Area of Interest (A slice of the Konkan Coast)
konkan_coast = ee.Geometry.Rectangle([73.75, 15.45, 73.85, 15.55])

# 3. Fetch clean Sentinel-2 imagery
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(konkan_coast) #take coordinates init. in kokan coast
         .filterDate('2025-01-01', '2025-12-31')#filter by last year
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))#the envt should not be cloudy so we filter out only those
         .median())

# 4. Create the automated Ground Truth Labels
# Water Mask (NDWI > 0)
ndwi = image.normalizedDifference(['B3', 'B8'])
water = ndwi.gt(0.0)

# Mangrove Mask (NDVI > 0.4 AND Not Water)
ndvi = image.normalizedDifference(['B8', 'B4'])
mangroves = ndvi.gt(0.4).And(water.Not())

# Combine: 0 = Sand/Background, 1 = Water, 2 = Mangroves
labels = ee.Image(0).where(water, 1).where(mangroves, 2).rename('label')

# 5. Stack our 4 input bands (RGB + NIR) with the label
training_stack = image.select(['B4', 'B3', 'B2', 'B8']).addBands(labels)

# 6. Export as TFRecord patches directly to your Google Drive
export_task = ee.batch.Export.image.toDrive(
    image=training_stack,
    description='Konkan_Coast_CNN_Data',
    folder='EarthEngine_Data', # It will create this folder in your Drive
    fileNamePrefix='coastal_patches',
    scale=10,
    region=konkan_coast,
    fileFormat='TFRecord',
    formatOptions={
        'patchDimensions': [256, 256],
        'compressed': True
    }
)

export_task.start()
print("🚀 Export task sent to Google's servers! Check your Google Drive in a few minutes.")

🚀 Export task sent to Google's servers! Check your Google Drive in a few minutes.


In [27]:
# Check the status of your most recent Earth Engine task
task = ee.batch.Task.list()[0]
print("Task Name:", task.config['description'])
print("Current Status:", task.status()['state'])

Task Name: Konkan_Coast_CNN_Data
Current Status: READY


In [28]:
# Cell 2: The TFRecord Data Pipeline (FINAL FIX)
import tensorflow as tf
import glob

file_pattern = '/content/drive/MyDrive/EarthEngine_Data/*.tfrecord.gz'
tfrecord_files = glob.glob(file_pattern)
print(f"Found {len(tfrecord_files)} training files.")

def parse_tfrecord(example_proto):
    feature_description = {
        'B4': tf.io.FixedLenFeature([256, 256], tf.float32),
        'B3': tf.io.FixedLenFeature([256, 256], tf.float32),
        'B2': tf.io.FixedLenFeature([256, 256], tf.float32),
        'B8': tf.io.FixedLenFeature([256, 256], tf.float32),

        # FIX 1: Tell TensorFlow to read the compressed label as a raw byte string
        'label': tf.io.FixedLenFeature([], tf.string),
    }

    parsed = tf.io.parse_single_example(example_proto, feature_description)

    # Stack the 4 optical bands
    image = tf.stack([parsed['B4'], parsed['B3'], parsed['B2'], parsed['B8']], axis=-1)

    # FIX 2: Decode the raw bytes, reshape into our map, and cast to integer
    # We use uint8 because Earth Engine typically compresses categorical labels into 8-bit unsigned integers
    label = tf.io.decode_raw(parsed['label'], out_type=tf.uint8)
    label = tf.reshape(label, [256, 256])
    label = tf.cast(label, tf.int32)

    return image, label

# Create the high-performance dataset
dataset = tf.data.TFRecordDataset(tfrecord_files, compression_type='GZIP')
dataset = dataset.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(16).prefetch(tf.data.AUTOTUNE)

print("✅ Data pipeline repaired! Bytes decoded. Fed to GPU.")

Found 1 training files.
✅ Data pipeline repaired! Bytes decoded. Fed to GPU.


In [29]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_coastal_unet(input_shape=(256, 256, 4), num_classes=3):
    inputs = layers.Input(shape=input_shape, name="satellite_input")

    # 4-channel to 3-channel adapter
    adapter = layers.Conv2D(filters=3, kernel_size=(1, 1), padding='same', name="band_adapter")(inputs)

    # Encoder: MobileNetV2
    backbone = tf.keras.applications.MobileNetV2(input_shape=(256, 256, 3), include_top=False, weights='imagenet')

    layer_names = [
        "block_1_expand_relu",   # 128x128
        "block_3_expand_relu",   # 64x64
        "block_6_expand_relu",   # 32x32
        "block_13_expand_relu",  # 16x16
        "block_16_project",      # 8x8
    ]

    skip_outputs = [backbone.get_layer(name).output for name in layer_names]
    backbone_model = models.Model(inputs=backbone.input, outputs=skip_outputs)

    skips = backbone_model(adapter)
    bottleneck = skips[-1]
    skips = reversed(skips[:-1])

    # Decoder
    x = bottleneck
    dims = [16, 32, 64, 128]

    for dim, skip in zip(dims, skips):
        x = layers.Conv2DTranspose(filters=skip.shape[-1], kernel_size=(2, 2), strides=(2, 2), padding='same')(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(filters=skip.shape[-1], kernel_size=(3, 3), padding='same', activation='relu')(x)
        x = layers.Conv2D(filters=skip.shape[-1], kernel_size=(3, 3), padding='same', activation='relu')(x)

    x = layers.Conv2DTranspose(filters=64, kernel_size=(2, 2), strides=(2, 2), padding='same')(x)

    # Output layer (3 classes: Sand, Water, Mangrove)
    outputs = layers.Conv2D(filters=num_classes, kernel_size=(1, 1), padding='same', activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs, name="Coastal_Erosion_UNet")

print("✅ Model architecture successfully loaded into memory!")

✅ Model architecture successfully loaded into memory!


In [30]:
# Cell 3: Compile and Train
from tensorflow.keras.callbacks import ModelCheckpoint

# 1. Compile the model
# We use sparse_categorical_crossentropy because our labels are integers (0, 1, 2)
model = build_coastal_unet()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 2. Create a callback to save the best weights automatically
checkpoint = ModelCheckpoint('/content/drive/MyDrive/EarthEngine_Data/coastal_unet_best.h5',
                             monitor='accuracy',
                             save_best_only=True,
                             mode='max')

# 3. START TRAINING
print("🚀 Launching GPU Training Sequence...")
history = model.fit(dataset, epochs=10, callbacks=[checkpoint])

print("🎉 Training Complete! Weights saved to Google Drive.")

/tmp/ipykernel_3391/293451673.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = tf.keras.applications.MobileNetV2(input_shape=(256, 256, 3), include_top=False, weights='imagenet')


🚀 Launching GPU Training Sequence...
Epoch 1/10
      1/Unknown 112s 112s/step - accuracy: 0.4085 - loss: 1.2197

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1/1 ━━━━━━━━━━━━━━━━━━━━ 114s 114s/step - accuracy: 0.4085 - loss: 1.2197
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 619ms/step - accuracy: 0.3496 - loss: 1.2425
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 538ms/step - accuracy: 0.4433 - loss: 1.0200

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.4433 - loss: 1.0200   
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 538ms/step - accuracy: 0.7075 - loss: 0.8663

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7075 - loss: 0.8663   
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 462ms/step - accuracy: 0.7156 - loss: 0.7513

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7156 - loss: 0.7513   
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - accuracy: 0.7283 - loss: 0.6361

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7283 - loss: 0.6361   
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 461ms/step - accuracy: 0.7348 - loss: 0.5491

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7348 - loss: 0.5491   
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.7533 - loss: 0.4852

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.7533 - loss: 0.4852   
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 583ms/step - accuracy: 0.7501 - loss: 0.4781
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 466ms/step - accuracy: 0.7332 - loss: 0.5350
🎉 Training Complete! Weights saved to Google Drive.
